# 5) Forecasts in operations: what errors cost, and what forecasts are worth

## Objectives

At the end of this notebook you should be able to:

- turn forecast errors into **money** with an imbalance-settlement model, and value an improvement in € per year
- explain why, when shortfalls cost more than surpluses, the best schedule is a **quantile** of the forecast, not the forecast itself
- build a **fundamental price forecast** from a merit order and forecasts of demand and solar
- measure what a price forecast is worth to a **battery**, and why the most accurate forecast is not always the most valuable

Lessons 1–4 measured forecasts in MW and €/MWh of error. Nobody pays for MAE. A supplier pays for imbalances, a battery earns from spreads, a TSO pays for reserves. This lesson closes the loop to Modules 02 and 03: the forecasts of this module feed decisions, and decisions are what make a forecast valuable — or not. All prices and penalties are **illustrative**.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor

from forecast_data import load_data, market_price, merit_order_price, solar_data, temperature_forecast

warnings.filterwarnings("ignore")
logging.getLogger("sklearn").setLevel(logging.ERROR)

### Three demand forecasts from this module

In [2]:
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
idx, dow = df.index, df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = idx.hour, idx.month
demand = df.demand
FORMULA = "demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
honest = df.assign(temperature=df.temperature_forecast)
regression = smf.ols(FORMULA, df.loc[:"2024"]).fit().predict(honest)                       # lesson 2

daytype_code = np.where(df.holiday | (dow == 6), 2, np.where(dow == 5, 1, 0))
issue = idx.normalize() - pd.Timedelta(hours=13)
X = pd.DataFrame({"hour": idx.hour, "daytype": daytype_code, "day_of_year": idx.dayofyear,
                  "temperature_forecast": df.temperature_forecast, "last_known": demand.reindex(issue).to_numpy(),
                  "same_hour_2_days_ago": demand.shift(48), "same_hour_last_week": demand.shift(168)}, index=idx)
usable = X.notna().all(axis=1)
boost = HistGradientBoostingRegressor(max_iter=1500, learning_rate=0.02, random_state=0)
boost.fit(X[usable & (idx.year < 2025)], demand[usable & (idx.year < 2025)])
boosting = pd.Series(boost.predict(X[usable]), index=idx[usable]).reindex(idx)

forecasts = {"same hour last week (lesson 1)": demand.shift(168), "regression (lesson 2)": regression,
             "regression + boosting ensemble (lesson 3)": 0.5 * regression + 0.5 * boosting}
y = demand.loc["2025"]
pd.Series({name: float((f.loc["2025"] - y).abs().mean()) for name, f in forecasts.items()}, name="MAE_2025_MW").round(1)

same hour last week (lesson 1)               142.5
regression (lesson 2)                         73.6
regression + boosting ensemble (lesson 3)     68.8
Name: MAE_2025_MW, dtype: float64

### What an error costs: imbalance settlement

A supplier buys its forecast demand in the day-ahead market. In real time every MWh it is off is settled as an **imbalance**: if its customers use more than it bought, it pays for the shortfall at a premium; if they use less, it sells the surplus back at a discount. Take a premium of **60 €/MWh** for being short and a discount of **40 €/MWh** for being long — relative to the day-ahead price, and illustrative. Real imbalance prices vary hour by hour and by system.

In [3]:
SHORT, LONG = 60.0, 40.0                                   # EUR per MWh short / long, relative to the day-ahead price


def imbalance_cost(schedule, actual=y):
    """Imbalance cost (EUR) of buying `schedule` when `actual` is consumed."""
    gap = actual - schedule.loc[actual.index]
    return float((SHORT * gap.clip(lower=0) + LONG * (-gap).clip(lower=0)).sum())


costs = pd.DataFrame({name: {"MAE_MW": float((f.loc["2025"] - y).abs().mean()), "imbalance_cost_MEUR": imbalance_cost(f) / 1e6}
                      for name, f in forecasts.items()}).T
costs.loc["perfect forecast"] = [0.0, 0.0]
saving = costs.loc["regression (lesson 2)", "imbalance_cost_MEUR"] - costs.loc["regression + boosting ensemble (lesson 3)", "imbalance_cost_MEUR"]
print(f"from regression to ensemble: {saving:.2f} MEUR per year saved")
costs.round(2)

from regression to ensemble: 1.76 MEUR per year saved


                                           MAE_MW  imbalance_cost_MEUR
same hour last week (lesson 1)             142.50                62.51
regression (lesson 2)                       73.56                32.09
regression + boosting ensemble (lesson 3)   68.76                30.33
perfect forecast                             0.00                 0.00

The imbalance bill follows the MAE almost exactly: about **0.44 M€ per year for every MW** of average error. That is no coincidence — with errors evenly split between short and long, each MWh of error costs on average (60 + 40) / 2 = 50 €, and a year has 8,760 hours: 50 × 8,760 = 0.44 M€ per MW. The regression of lesson 2 halves the bill of lesson 1's weekly baseline, from 62.5 to 32.1 M€ a year, and lesson 3's ensemble saves another **1.76 M€ a year** — the money a supplier could spend on better models, better weather data or a forecasting team. Most of the 30 M€ that remain cannot be removed by a better demand model: even with tomorrow's actual temperature, lesson 2's model keeps an error of 58.6 MW — about 26 M€ a year at this rule of thumb — which is demand's own unpredictable noise. Only the rest is the temperature forecast's share.

### When shortfalls cost more: schedule a quantile

The forecast is the best guess of demand, but it is not the best amount to **buy**. Buying one extra MWh costs 40 € if it turns out not to be needed and saves 60 € if it is. It pays to keep buying until the chance of needing the next MWh drops to 40 / (40 + 60) = 0.4 — that is, to buy the **60 % quantile** of demand, not the median or the mean:

α* = short premium / (short premium + long discount) = 60 / 100 = 0.6

This is the classic **newsvendor** result, and it is where lesson 4's quantiles turn into a decision. Add the α-quantile of the regression's honest 2024 errors to its forecast and compare the cost for a range of α:

In [4]:
errors_2024 = (demand - smf.ols(FORMULA, df.loc["2023"]).fit().predict(honest)).loc["2024"]   # lesson 4's honest errors
alphas = np.round(np.arange(0.3, 0.81, 0.05), 2)
by_alpha = pd.Series({a: imbalance_cost(regression + np.quantile(errors_2024, a)) / 1e6 for a in alphas}, name="cost_MEUR")
point_cost = imbalance_cost(regression) / 1e6
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(by_alpha.index, by_alpha.to_numpy(), marker="o")
ax.axhline(point_cost, color="grey", ls=":", label="buy the point forecast")
ax.axvline(0.6, color="black", ls="--", lw=1, label="α* = 0.6")
ax.set(xlabel="quantile α bought", ylabel="imbalance cost 2025 (MEUR)", title="Buying a quantile instead of the forecast")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"point forecast {point_cost:.2f} MEUR; alpha = 0.6: {by_alpha[0.6]:.2f} MEUR; "
      f"lowest on 2025 at alpha = {by_alpha.idxmin()}: {by_alpha.min():.2f} MEUR")

point forecast 32.09 MEUR; alpha = 0.6: 31.32 MEUR; lowest on 2025 at alpha = 0.65: 30.99 MEUR


Buying the **60 % quantile** instead of the forecast cuts the bill from 32.09 to 31.32 M€ — almost half of what the whole ensemble of lesson 3 saved, for the price of adding one number to the schedule. The curve is flat near its minimum: on 2025 the lowest cost lies at α = 0.65 (30.99 M€), a little beyond the theoretical 0.6, because 2025's errors are not exactly 2024's. Two warnings sit in the figure. Buying too little is expensive — at α = 0.3 the bill rises to 43.7 M€, because every shortfall costs 60 € — and even α = 0.5 costs **more** than the point forecast (33.5 against 32.1 M€): the median of 2024's errors is −12 MW, a slight over-forecast of 2024 that did not repeat in 2025, when demand ran a little higher. The lesson: when costs are asymmetric, the right question is not "what will demand be?" but "what should I buy, given what I know about my errors?" — and lesson 4's honest quantiles answer it.

### Forecasting the price

Day-ahead prices follow the **merit order** (M2.1): the price is the cost of the last block needed to cover demand that wind, solar and must-run plants do not. `forecast_data.py` builds an illustrative Croatian-style supply stack — must-run renewables, a nuclear share, reservoir hydro offered at its water value (Module 03, lesson 7), gas, coal, imports, peakers — and sets each hour's price by the residual demand (demand minus solar) plus small persistent deviations.

A **fundamental** price forecast does the same with **forecasts**: residual demand = demand forecast − solar forecast, priced through the same merit order. Compare it with two price-only baselines:

In [5]:
solar = solar_data(df)
price = market_price(demand - solar.solar)
fundamental = pd.Series(merit_order_price(regression - solar.solar_forecast), index=idx)
p = price.loc["2025"]
hour_avg_2024 = price.loc["2024"].groupby(price.loc["2024"].index.hour).mean()
price_forecasts = {"fundamental (merit order of forecasts)": fundamental.loc["2025"],
                   "average price of each hour in 2024": pd.Series(hour_avg_2024.loc[p.index.hour].to_numpy(), index=p.index),
                   "same hour last week": price.shift(168).loc["2025"]}
price_forecasts["50/50 blend: fundamental + hourly average"] = (0.5 * price_forecasts["fundamental (merit order of forecasts)"]
                                                               + 0.5 * price_forecasts["average price of each hour in 2024"])
# the same fundamental forecast, with ties between hours broken by a negligible share of the hourly average
price_forecasts["fundamental, ties broken"] = (price_forecasts["fundamental (merit order of forecasts)"]
                                               + 1e-6 * price_forecasts["average price of each hour in 2024"])
print(f"2025 prices: mean {p.mean():.1f} EUR/MWh, average daily spread {p.groupby(p.index.date).agg(lambda s: s.max() - s.min()).mean():.1f} EUR/MWh")
pd.Series({name: float((f - p).abs().mean()) for name, f in price_forecasts.items()}, name="MAE_EUR_per_MWh").round(1)

2025 prices: mean 84.0 EUR/MWh, average daily spread 35.1 EUR/MWh


fundamental (merit order of forecasts)        6.4
average price of each hour in 2024           11.6
same hour last week                          10.0
50/50 blend: fundamental + hourly average     7.9
fundamental, ties broken                      6.4
Name: MAE_EUR_per_MWh, dtype: float64

### What a price forecast is worth: a battery

A 100 MW / 400 MWh battery with 85 % round-trip efficiency trades once a day: it charges in the four hours the forecast expects to be cheapest and discharges in the four it expects to be dearest, earning the **actual** prices. With a sensible rule it trades only when the forecast spread covers its losses — when 0.85 × (forecast price of the four dear hours) exceeds the forecast price of the four cheap ones.

In [6]:
fundamental_2025 = price_forecasts["fundamental (merit order of forecasts)"]
per_day = fundamental_2025.groupby(fundamental_2025.index.date)
print(f"fundamental forecast: {per_day.nunique().mean():.1f} distinct values per day on average; "
      f"{per_day.apply(lambda s: s.duplicated(keep=False).mean()).mean():.1%} of hours share their value with another hour")


def battery_profit(signal, actual=p, rule=True):
    """Profit (EUR) and trading days of a daily 4-hour arbitrage driven by `signal`, settled at `actual`."""
    profit, days = 0.0, 0
    for _, day in pd.DataFrame({"signal": signal, "actual": actual}).groupby(actual.index.date):
        order = day.sort_values("signal")
        cheap, dear = order.index[:4], order.index[-4:]
        if rule and 0.85 * day.loc[dear, "signal"].mean() <= day.loc[cheap, "signal"].mean():
            continue
        days += 1
        profit += 100 * (0.85 * day.loc[dear, "actual"].sum() - day.loc[cheap, "actual"].sum())
    return profit, days


def within_day_rank(signal, actual=p):
    """Mean correlation between forecast and actual ranks of the hours within each day."""
    frame = pd.DataFrame({"s": signal, "a": actual})
    return float(frame.groupby(actual.index.date).apply(lambda g: g.s.rank().corr(g.a.rank())).mean())


rows = {"perfect foresight": {"MAE_EUR_per_MWh": 0.0, "rank_corr": 1.0,
                              "profit_naive_MEUR": battery_profit(p, rule=False)[0] / 1e6,
                              "profit_with_rule_MEUR": battery_profit(p)[0] / 1e6, "trading_days": battery_profit(p)[1]}}
for name, f in price_forecasts.items():
    with_rule = battery_profit(f)
    rows[name] = {"MAE_EUR_per_MWh": float((f - p).abs().mean()), "rank_corr": within_day_rank(f),
                  "profit_naive_MEUR": battery_profit(f, rule=False)[0] / 1e6,
                  "profit_with_rule_MEUR": with_rule[0] / 1e6, "trading_days": with_rule[1]}
battery = pd.DataFrame(rows).T
battery.round(2)

fundamental forecast: 2.7 distinct values per day on average; 99.3% of hours share their value with another hour


                                           MAE_EUR_per_MWh  rank_corr  \
perfect foresight                                     0.00       1.00   
fundamental (merit order of forecasts)                6.42       0.56   
average price of each hour in 2024                   11.59       0.44   
same hour last week                                  10.04       0.32   
50/50 blend: fundamental + hourly average             7.89       0.49   
fundamental, ties broken                              6.42       0.53   

                                           profit_naive_MEUR  \
perfect foresight                                       2.02   
fundamental (merit order of forecasts)                  0.50   
average price of each hour in 2024                      0.60   
same hour last week                                    -0.16   
50/50 blend: fundamental + hourly average               0.76   
fundamental, ties broken                                0.81   

                                       

Accuracy and value part ways here. With perfect foresight the battery would earn 2.10 M€ a year; the forecasts earn between 0.20 and 1.29 M€, and their order is **not** the order of their MAE.

**The decision rule matters as much as the forecast.** Trading every day regardless, the fundamental forecast earns only 0.50 M€; with the rule — trade only when the forecast spread covers the 15 % losses — it earns 0.98 M€ on 199 days, because it now sits out the days where its own forecast says there is nothing to gain. The hourly average gains nothing from the rule: its spread always looks wide enough, so it trades all 365 days.

**Ties cost money.** The fundamental forecast can only take the prices of the stack's blocks — on average 2.7 distinct values a day, and 99.3 % of hours share their forecast with another hour. When, for example, four hours must be chosen among eight that look identical, the choice is arbitrary. Breaking those ties with a negligible share of the hourly average leaves the MAE unchanged at 6.42 €/MWh and lifts the profit from 0.98 to **1.29 M€** — 32 % more from the same accuracy. The 50/50 blend does something similar (1.20 M€) at a higher MAE (7.9 €/MWh). Even the within-day rank correlation misleads: it is slightly lower for the tie-broken forecast (0.53 against 0.56) although that forecast earns more.

The general rule: judge a forecast by the decision it feeds. A battery needs the right hours in the right order, a supplier needs the right quantile, a TSO needs the tails — and each of these can disagree with MAE.

## Check your understanding

**Imbalances**

1. Derive α* = 0.6 in your own words: what is gained and lost by buying one MWh more than the median?
2. If the short premium rose to 150 €/MWh and the long discount stayed at 40 €/MWh, which quantile should the supplier buy? Would it buy more or less than now?

**Value**

1. The fundamental forecast and its tie-broken twin have the same MAE but earn very different amounts. What exactly did breaking the ties change in the battery's decisions, and what does that say about judging a forecast by its error alone?
2. Name one decision in Modules 02 or 03 where a forecast's **bias** matters more than its MAE, and one where its **ranking of hours** matters most.